# Lab 3: Credit Card Fraud Detection

**Task:** detect fraudulent card transactions.

**Dataset:** European card transactions, September 2013 (Kaggle: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).

> **The data is not in this repository: `creditcard.csv` is about 150 MB, too big for git.** Download it from the link above (a free Kaggle account is needed), unzip it, and put `creditcard.csv` in this lab's `data/` folder.

The 284,807 transactions have 30 features: `Time` (seconds since the first transaction in the data), `Amount`, and `V1` to `V28`. For confidentiality, the bank released `V1`–`V28` only after transforming the original features with PCA (principal component analysis), so they have no business meaning. `Class` is 1 for fraud.

**What we will do:**
1. Look at the class imbalance.
2. Split into training, validation and test sets with stratification.
3. Scale the features without leaking information from the validation and test data.
4. Train logistic regression and SVMs, without and with imbalance handling (class weights, undersampling).
5. Compare them with precision, recall, F1, ROC curves and AUC; choose the cutoff that maximises the KS statistic.
6. Report the confusion matrix of the best model and discuss the trade-off between catching fraud and raising false alarms.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
cards = pd.read_csv("data/creditcard.csv")
print(cards.shape)
cards.head(3)

## 1. Class imbalance

In [ ]:
counts = cards["Class"].value_counts()
print(counts.to_dict())
print(f"fraud rate: {counts[1] / len(cards):.4%}  ->  one fraud per {counts[0] / counts[1]:.0f} normal transactions")

Only 0.17% of transactions are fraud. A "model" that says *no transaction is ever fraud* is 99.83% accurate and completely useless. **Accuracy is misleading here**; what matters is how many frauds we catch (**recall**) and how many of our alerts are real frauds (**precision**).

## 2. Train, validation and test sets

- **Training set (60%)**: fit the models.
- **Validation set (20%)**: choose between models and choose the cutoff.
- **Test set (20%)**: one final, honest evaluation of the chosen model.

**Stratification** keeps the fraud rate the same in all three. With only 492 frauds, a random split could easily leave one set with noticeably fewer.

In [ ]:
from sklearn.model_selection import train_test_split

X = cards.drop(columns="Class")
y = cards["Class"]
x_trainval, x_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
x_train, x_val, y_train, y_val = train_test_split(x_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=0)
for name, t in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name:10s}: {len(t):6d} transactions, {t.sum():3d} frauds ({t.mean():.4%})")

## 3. Preprocessing without leakage

`Amount` is very skewed (most transactions are small, a few are huge), so we take `log(1 + Amount)`. Then all features are standardised. The scaler learns means and standard deviations, so it must learn them **from the training data only**, and then apply those same numbers to the validation and test data. Putting it inside a pipeline guarantees this; fitting it on all the data first would let the test set influence the training.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer

log_amount = FunctionTransformer(np.log1p)
preprocess = ColumnTransformer([("amount", Pipeline([("log", log_amount), ("scale", StandardScaler())]), ["Amount"]),
                                ("rest", StandardScaler(), [c for c in X.columns if c != "Amount"])])

## 4. Models

We compute a **score** for each validation transaction (a probability for logistic regression, a decision value for SVMs); a higher score means "more likely fraud". At the default cutoff (probability 0.5, decision value 0) we compute precision, recall and F1; AUC measures the ranking over all cutoffs.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, f1_score, roc_curve,
                             average_precision_score, confusion_matrix)

def scores_of(model, x):
    return model.predict_proba(x)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(x)

results, val_scores = [], {}
def evaluate(name, model, x_fit, y_fit, default_cutoff):
    model.fit(x_fit, y_fit)
    s = scores_of(model, x_val)
    val_scores[name] = s
    pred = (s >= default_cutoff).astype(int)
    results.append({"model": name, "precision": precision_score(y_val, pred, zero_division=0),
                    "recall": recall_score(y_val, pred), "F1": f1_score(y_val, pred),
                    "ROC AUC": roc_auc_score(y_val, s), "PR AUC": average_precision_score(y_val, s)})
    return model

evaluate("logistic", Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000))]), x_train, y_train, 0.5)
evaluate("logistic, balanced weights", Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
         x_train, y_train, 0.5)
pd.DataFrame(results).round(3)

Without adjustment, logistic regression is precise but misses a large share of the frauds at the 0.5 cutoff. Balanced class weights make each fraud count about 290 times as much as a normal transaction in the loss: recall jumps, and precision collapses, because the probabilities are pushed up for everything. The **ROC AUC doesn't improve** (it is even slightly lower, 0.981 against 0.993): the weighted model doesn't rank transactions better; class weights mostly move the default cutoff.

### Undersampling
Keep all training frauds and a random sample of the normal transactions, here 10 normal for each fraud. Only the **training** data is undersampled; validation and test keep the real fraud rate, otherwise their precision would describe a world where 9% of transactions are fraud.

In [ ]:
rng = np.random.default_rng(0)
fraud_idx = y_train[y_train == 1].index
normal_idx = rng.choice(y_train[y_train == 0].index, size=10 * len(fraud_idx), replace=False)
under_idx = fraud_idx.append(pd.Index(normal_idx))
x_under, y_under = x_train.loc[under_idx], y_train.loc[under_idx]
print("undersampled training set:", len(y_under), "rows, fraud share", round(y_under.mean(), 3))

evaluate("logistic, undersampled 1:10", Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000))]), x_under, y_under, 0.5)
pd.DataFrame(results).round(3)

### SVMs

A **linear SVM** finds a separating hyperplane with the widest margin (hinge loss plus an L2 penalty). It scales well to large data. An **RBF-kernel SVM** can draw curved boundaries, but its training time grows roughly between $n^2$ and $n^3$: on 170,000 transactions it would take very long. Undersampling solves that too: on the 5,000-row undersampled set it trains in seconds.

In [ ]:
import time

t = time.time()
evaluate("linear SVM, balanced weights", Pipeline([("prep", preprocess), ("model", LinearSVC(C=0.01, class_weight="balanced", max_iter=5000))]),
         x_train, y_train, 0.0)
print(f"linear SVM on all {len(y_train)} training rows: {time.time() - t:.0f}s")
t = time.time()
evaluate("RBF SVM, undersampled 1:10", Pipeline([("prep", preprocess), ("model", SVC(kernel="rbf", C=1, gamma="scale"))]),
         x_under, y_under, 0.0)
print(f"RBF SVM on {len(y_under)} undersampled rows: {time.time() - t:.0f}s")
summary = pd.DataFrame(results).round(3)
summary

## 5. ROC curves and the KS-optimal cutoff

The default cutoff is arbitrary. The **KS statistic** is the largest gap between the true positive rate (share of frauds caught) and the false positive rate (share of normal transactions flagged) over all cutoffs; the cutoff where it is reached separates the two classes best. We choose it on the **validation** set.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ks_rows = []
for name, s in val_scores.items():
    fpr, tpr, thr = roc_curve(y_val, s)
    k = np.argmax(tpr - fpr)
    ks_rows.append({"model": name, "KS": tpr[k] - fpr[k], "cutoff": thr[k], "TPR (recall)": tpr[k], "FPR": fpr[k]})
    ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_val, s):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate (recall)")
ax.legend(fontsize=8); plt.show()
ks_table = pd.DataFrame(ks_rows).sort_values("KS", ascending=False)
ks_table.round(4)

All the curves hug the top-left corner: ROC AUC is above 0.97 for every model. With so many normal transactions, even a false positive rate of 1% means about 570 false alarms in the validation set, more than the 99 frauds in it. That is why the **precision-recall view** (PR AUC in the table above) is more informative for rare events: it shows how many alerts are real.

## 6. The best model on the test set

We take the model with the highest validation KS, apply its validation-chosen cutoff to the **test** set, and look at the confusion matrix.

In [ ]:
best_name = ks_table.iloc[0]["model"]
best_cutoff = ks_table.iloc[0]["cutoff"]
print("best model by validation KS:", best_name, "| cutoff", round(best_cutoff, 4))

refit = {"logistic": (Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000))]), x_train, y_train),
         "logistic, balanced weights": (Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))]), x_train, y_train),
         "logistic, undersampled 1:10": (Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000))]), x_under, y_under),
         "linear SVM, balanced weights": (Pipeline([("prep", preprocess), ("model", LinearSVC(C=0.01, class_weight="balanced", max_iter=5000))]), x_train, y_train),
         "RBF SVM, undersampled 1:10": (Pipeline([("prep", preprocess), ("model", SVC(kernel="rbf", C=1, gamma="scale"))]), x_under, y_under)}
model, xf, yf = refit[best_name]
model.fit(xf, yf)
s_test = scores_of(model, x_test)
pred_test = (s_test >= best_cutoff).astype(int)
cm = confusion_matrix(y_test, pred_test)
print(pd.DataFrame(cm, index=["actual normal", "actual fraud"], columns=["predicted normal", "predicted fraud"]))
fpr, tpr, _ = roc_curve(y_test, s_test)
print(f"test: ROC AUC {roc_auc_score(y_test, s_test):.4f}, KS {np.max(tpr - fpr):.4f}, "
      f"precision {precision_score(y_test, pred_test):.3f}, recall {recall_score(y_test, pred_test):.3f}, F1 {f1_score(y_test, pred_test):.3f}")

### The trade-off: catching fraud vs false alarms

The top models are practically tied on validation KS (0.9485 against 0.9476), so which one "wins" is partly luck with only 99 validation frauds; the test KS (0.858) is lower than the validation KS of the winner for the same reason: picking the best of five on a small number of frauds is slightly optimistic.

At the KS cutoff the model catches 85 of the 98 test frauds and raises 621 false alarms: precision 12%. The KS cutoff maximises *recall minus false positive rate*, which treats a missed fraud and a false alarm as equally bad once each is expressed as a rate. With 578 normal transactions per fraud, that puts the cutoff low. Moving the cutoff up raises precision and lowers recall:

In [ ]:
rows = []
for c in np.quantile(s_test, [0.95, 0.99, 0.995, 0.998, 0.999]):
    p = (s_test >= c).astype(int)
    rows.append({"flag top %": round(100 * p.mean(), 2), "alerts": p.sum(), "frauds caught": int((p & y_test.values).sum()),
                 "recall": recall_score(y_test, p), "precision": precision_score(y_test, p)})
pd.DataFrame(rows).round(3)

- **Recall** answers "of all frauds, how many did we catch?" Every missed fraud is money lost and an angry customer.
- **Precision** answers "of all the transactions we blocked or queried, how many were really fraud?" Every false alarm is a declined card at a checkout, a call to a customer, and work for the fraud team.

No cutoff is "correct"; it depends on what each kind of mistake costs. If a missed fraud costs on average ₹10,000 and a false alarm costs ₹100 of customer annoyance and staff time, catching one more fraud is worth up to 100 false alarms, and a low cutoff is right. If the fraud team can only review 200 cases a day, the cutoff is set by that capacity. In practice banks often use two cutoffs: very high scores are blocked automatically, a middle band is reviewed or confirmed with the customer by SMS.

**Summary of the comparison:** the models rank transactions similarly well (validation ROC AUC 0.98–0.99). At its default cutoff the RBF SVM trained on undersampled data has the best balance (F1 0.79). Class weights and undersampling change the default cutoff far more than the ranking; once the cutoff is chosen properly (by KS here), the differences shrink. Undersampling makes kernel SVMs feasible on large data and trains much faster, at the cost of using only part of the normal transactions.